# Physics-aware vs statistical anomaly detection in solar PV
Exactly the pipeline from the slides: simulated single-diode PV environment (P = V x I) -> labeled dataset -> LSTM autoencoder -> FGSM/PGD vs physics-projected PGD -> comparison.
The simulator is driven by the REAL irradiance and temperature in your CSV files. Run -> Run All Cells.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
torch.manual_seed(0); rng = np.random.default_rng(0)

# >>> change this if your CSVs are somewhere else <<<
DATA_DIR = r"C:\Users\Shreya\Downloads"
TRAIN_CSV = os.path.join(DATA_DIR, "pv_train_merged.csv")
TEST_CSV  = os.path.join(DATA_DIR, "pv_test_merged.csv")
F = ["G", "T", "V", "I", "P"]
L, S = 24, 6          # window length, stride

## 1-2. PV environment, fault injection, dataset

In [ ]:
# ===== 1. SIMULATED PV ENVIRONMENT: single-diode model, P = V x I enforced =====
ISC, VOC, NS, NID = 8.2, 37.0, 60, 1.3
def a_of(T):  return NID * NS * 0.02585 * (T + 273.15) / 298.15
def i0_of(T): return ISC / (np.exp(VOC / a_of(25.0)) - 1) * np.exp(0.065 * (T - 25))
def iph(G, T): return ISC * (G / 1000) * (1 + 0.0005 * (T - 25))
def i_curve(V, G, T):
    return np.maximum(iph(G, T) - i0_of(T) * (np.exp(np.minimum(V / a_of(T), 50)) - 1), 0)
def mpp(G, T):
    Vg = np.linspace(0, VOC, 300)[None, :]
    I = i_curve(Vg, G[:, None], T[:, None]); P = Vg * I
    k = P.argmax(1)
    return Vg[0][k], I[np.arange(len(k)), k]

def simulate_from_weather(path):
    """Real irradiance + temperature from YOUR csv  ->  single-diode model  ->  V, I, P."""
    d = pd.read_csv(path, usecols=["instantaneous_global_radiation", "clearsky_Tamb", "radiation_mean", "radiation_std",
                                   "tprt_mean", "tprt_std", "label", "site_num", "date", "source_file"])
    d = d[d.label == 0]                                   # normal rows only = clean weather
    G = np.clip(d.instantaneous_global_radiation * d.radiation_std + d.radiation_mean, 0, None).to_numpy()   # W/m2
    Ta = (d.clearsky_Tamb * d.tprt_std + d.tprt_mean).to_numpy()                                           # deg C
    T = Ta + 0.03 * G                                     # cell temperature
    V, I = np.zeros(len(G)), np.zeros(len(G))
    for s in range(0, len(G), 4000):
        V[s:s+4000], I[s:s+4000] = mpp(G[s:s+4000], T[s:s+4000])
    V *= 1 + .005 * rng.standard_normal(len(G)); I *= 1 + .005 * rng.standard_normal(len(G))
    P = V * I * (1 + .003 * rng.standard_normal(len(G)))
    X = np.stack([G, T, V, I, P], 1).astype(np.float32)
    keys = d.groupby(["source_file", "site_num", "date"], sort=False).ngroup().to_numpy()
    return X, keys, d.site_num.to_numpy()

# ===== 2. FAULT INJECTION: partial shading, sensor drift, inverter fault =====
FAULTS = {1: "partial_shading", 2: "sensor_drift", 3: "inverter_fault"}
def inject(X):
    X = X.copy(); ft = np.zeros(len(X), int)
    if len(X) < 40: return X, ft
    for _ in range(len(X) // 40):
        s = rng.integers(0, len(X) - 20); e = s + rng.integers(8, 20); kind = int(rng.integers(1, 4))
        if kind == 1:   # shading: current and voltage drop, power consistent
            X[s:e, 3] *= rng.uniform(.4, .7); X[s:e, 2] *= rng.uniform(.9, .97); X[s:e, 4] = X[s:e, 2] * X[s:e, 3]
        elif kind == 2: # V sensor drifts, P not updated
            X[s:e, 2] *= 1 + np.linspace(0, rng.uniform(.1, .25), e - s)
        else:           # inverter fault
            X[s:e, 3] *= rng.uniform(0, .2); X[s:e, 4] = X[s:e, 2] * X[s:e, 3]
        ft[s:e] = kind
    return X, ft

def build(X, keys, site, faults):
    W, T, SI = [], [], []
    cut = np.flatnonzero(np.diff(keys)) + 1
    for xs, st in zip(np.split(X, cut), np.split(site, cut)):
        ft = np.zeros(len(xs), int)
        if faults: xs, ft = inject(xs)
        for i in range(0, len(xs) - L + 1, S):
            w = xs[i:i+L]; f = ft[i:i+L]; nf = (f > 0).sum()
            if w[:, 0].mean() < 150 or 0 < nf < 4: continue      # skip night windows and barely-faulty windows
            W.append(w); T.append(f.max()); SI.append(st[0])
    return np.stack(W), np.array(T), np.array(SI)

Xs, ks, ss = simulate_from_weather(TRAIN_CSV)
Xtr, _, str_ = build(Xs, ks, ss, faults=False)
Xs, ks, ss = simulate_from_weather(TEST_CSV)
Xte, tte, _ = build(Xs, ks, ss, faults=True)
yte = (tte > 0).astype(int)
sites = np.unique(str_); isval = np.isin(str_, sites[-3:])
Xfit, Xval = Xtr[~isval], Xtr[isval]
print("train windows:", len(Xfit), "| val:", len(Xval), "| test:", len(Xte), "(faulty:", yte.sum(), ")")
print("check P = V*I on clean data (mean rel. error):", float(np.mean(np.abs(Xfit[..., 4] - Xfit[..., 2] * Xfit[..., 3]) / (Xfit[..., 4] + 1))))

rows = Xfit.reshape(-1, 5); span = rows.max(0) - rows.min(0)
LO, HI = rows.min(0) - .05 * span, rows.max(0) + .05 * span
to_norm = lambda x: np.clip((x - LO) / (HI - LO), 0, 1).astype(np.float32)
to_raw  = lambda xn: xn * (HI - LO) + LO

# ===== physics rules (used by the physics-constrained attack and by the physics check) =====
TOL = 0.15
def project(xn):
    """Project onto the single-diode physics: V,G >= 0, I near the I-V curve for (V,G,T), then P = V*I."""
    r = to_raw(xn)
    G, T, V = np.maximum(r[..., 0], 0), r[..., 1], np.clip(r[..., 2], 0, VOC)
    Ic = i_curve(V, G, T)
    I = np.clip(r[..., 3], Ic * (1 - TOL), Ic * (1 + TOL))
    return to_norm(np.stack([G, T, V, I, V * I], -1))

def physical_ok(xn, slack=0.05):
    r = to_raw(xn)
    G, T, V, I, P = [r[..., k] for k in range(5)]
    Ic = i_curve(V, G, T)
    c1 = np.abs(P - V * I) / (np.abs(P) + 1) < 0.02
    c2 = np.abs(I - Ic) / (Ic + .1) < TOL + slack
    return (c1 & c2).all(1)

## 3. Train the LSTM autoencoder (normal data only)

In [ ]:
class AE(nn.Module):
    def __init__(s, f=5, h=32):
        super().__init__()
        s.enc = nn.LSTM(f, h, batch_first=True); s.dec = nn.LSTM(h, h, batch_first=True); s.out = nn.Linear(h, f)
    def forward(s, x):
        _, (h, _) = s.enc(x)
        z = h[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        return s.out(s.dec(z)[0])

def err(model, x):
    with torch.no_grad():
        x = torch.as_tensor(x); return ((model(x) - x) ** 2).mean((1, 2)).numpy()

model = AE(); opt = torch.optim.Adam(model.parameters(), 1e-3)
trn = torch.as_tensor(to_norm(Xfit)); losses = []
for ep in range(30):
    tot = 0
    for b in torch.randperm(len(trn)).split(64):
        loss = ((model(trn[b]) - trn[b]) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(b)
    losses.append(tot / len(trn))
    if ep % 5 == 0: print("epoch", ep, "loss", round(losses[-1], 5))
model.eval()
plt.plot(losses); plt.xlabel("epoch"); plt.ylabel("train MSE"); plt.title("Training loss"); plt.show()

## Test on clean data

In [ ]:
thr = np.percentile(err(model, to_norm(Xval)), 99)
Xt = to_norm(Xte)
e = err(model, Xt); flag_ae = e > thr
flag_phys = ~physical_ok(Xt)
print(f"LSTM-AE : detects {flag_ae[yte==1].mean():.2f} of faults | false alarms {flag_ae[yte==0].mean():.2f}")
print(f"Physics : detects {flag_phys[yte==1].mean():.2f} of faults | false alarms {flag_phys[yte==0].mean():.2f}")
names = np.array(["normal"] + [FAULTS[k] for k in (1, 2, 3)])[tte]
tab = pd.DataFrame({"type": names, "LSTM-AE": flag_ae, "Physics check": flag_phys})
tab = tab[yte == 1].groupby("type").mean().round(2); print(tab)
tab.plot.bar(figsize=(7, 3.5)); plt.ylabel("detection rate"); plt.title("Detection on clean (un-attacked) faults"); plt.tight_layout(); plt.show()

## 4a / 4b. Conventional vs physics-constrained attacks

In [ ]:
def attack(model, xn, eps, steps, alpha, physics):
    x0 = torch.as_tensor(xn); x = x0.clone()
    for _ in range(steps):
        x.requires_grad_(True)
        loss = ((model(x) - x) ** 2).mean((1, 2)).sum()
        g, = torch.autograd.grad(loss, x)
        x = x.detach() - alpha * g.sign()
        x = torch.min(torch.max(x, x0 - eps), x0 + eps).clamp(0, 1)
        if physics: x = torch.as_tensor(project(x.numpy()))
    return x.detach().numpy()

fa = np.where(yte == 1)[0]
A = Xt[rng.choice(fa, size=min(500, len(fa)), replace=False)]
print("attacking", len(A), "faulty windows")
res, epss = {}, [0.02, 0.05, 0.1, 0.2]
for eps in epss:
    for name, steps, phys in [("FGSM", 1, False), ("PGD", 20, False), ("PGD+physics", 20, True)]:
        adv = attack(model, A, eps, steps, eps if steps == 1 else eps / 5, phys)
        evade = (err(model, adv) < thr).mean(); ok = physical_ok(adv).mean()
        res[(name, eps)] = (evade, ok, adv)
        print(f"eps={eps:<5} {name:<12} evasion={evade:.2f} phys-valid={ok:.2f} caught-by-physics={1-ok:.2f}")

## 5. Comparison

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n in ["FGSM", "PGD", "PGD+physics"]:
    ax[0].plot(epss, [res[(n, e)][0] for e in epss], "o-", label=n)
    ax[1].plot(epss, [res[(n, e)][1] for e in epss], "o-", label=n)
ax[0].set_title("Evasion rate (attacker wins)"); ax[1].set_title("Fraction physically valid")
for a in ax: a.set_xlabel("epsilon"); a.set_ylim(-.05, 1.05); a.legend()
plt.tight_layout(); plt.show()

eps, i = 0.1, 0
fig, axs = plt.subplots(1, 5, figsize=(16, 3))
for k, a in enumerate(axs):
    a.plot(to_raw(A[i])[:, k], "k", lw=2, label="faulty (original)")
    for n in ["FGSM", "PGD", "PGD+physics"]: a.plot(to_raw(res[(n, eps)][2][i])[:, k], label=n)
    a.set_title(F[k])
axs[0].legend(fontsize=7); plt.tight_layout(); plt.show()